# Half_PSD — Platform Screen Door GraphRAG Demo

This notebook explores the Door Control Unit (DCU) and its relationships in the
**C152E Half-Height Platform Screen Door operation and maintenance manual**:
[`152E-GEN-8002(doc)-C.pdf`](data/152E-GEN-8002%28doc%29-C.pdf).

It uses the **Half_PSD** profile in `profiles.yaml`, `ontology.yaml`, and `prompt.yaml`.
The configured heading ranges cover system overview, DCU control, initialization
and degraded operation, DCU replacement, and DCU reset.

The demonstration follows source sections into text chunks, extracted entities
and relationships, community summaries, and answers about the DCU. Use the graph's
source excerpts to inspect the evidence behind an extracted relationship.


The implementation lives in `src/graph_rag_schema.py`, `src/graph_rag_engine.py`,
`src/graph_rag_services.py`, and `src/graph_rag_manager.py`. This notebook presents
the Half_PSD workflow using those reusable modules.


---
## 0. Install Dependencies

In [ ]:
# %pip install -r requirements.txt

---
## 1. Load the Half_PSD Profile

This notebook uses `PROFILE = "Half_PSD"`. Its ontology and prompts are selected
before importing `src`, overriding the default ontology for this Python process.

Edit the `Half_PSD` settings in the YAML files to adjust section ranges, chunking,
models, or questions. **Restart the kernel and run from the top after YAML or
Python code changes.** The notebook does not rewrite the YAML files.


In [ ]:
import os
import sys
from pathlib import Path

PROFILE = "Half_PSD"  # This notebook is dedicated to the Half_PSD manual

loaded_schema = sys.modules.get("src.graph_rag_schema")
if loaded_schema is not None and getattr(loaded_schema, "ACTIVE_ONTOLOGY", None) != PROFILE:
    raise RuntimeError("Profile changed after import. Restart the kernel and run from the top.")

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ontology.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT / "causalRAG"
if not (PROJECT_ROOT / "ontology.yaml").is_file():
    raise FileNotFoundError("Start the notebook from causalRAG or its parent directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from profile_config import load_profiles

PROFILES = load_profiles()

if PROFILE not in PROFILES:
    raise ValueError("profiles.yaml must define the Half_PSD profile.")

os.environ["GRAPH_RAG_PROFILE"] = PROFILE
# Load all settings from the selected profile together.
PROFILE_CONFIG = PROFILES[PROFILE]
INPUT_TYPE = PROFILE_CONFIG["input_type"]
DATASET_FILE = PROJECT_ROOT / PROFILE_CONFIG["input_file"]
PDF_PAGES = PROFILE_CONFIG.get("pages")
PDF_SECTIONS = PROFILE_CONFIG.get("sections")
PDF_SECTION_RANGES = PROFILE_CONFIG.get("section_ranges")
if DATASET_FILE.resolve() != (PROJECT_ROOT / "data/152E-GEN-8002(doc)-C.pdf").resolve():
    raise ValueError("Half_PSD.input_file must point to data/152E-GEN-8002(doc)-C.pdf.")
if not PDF_SECTION_RANGES:
    raise ValueError("Configure Half_PSD.section_ranges in profiles.yaml before running this demo.")
PDF_CHUNK_SIZE = PROFILE_CONFIG["pdf_chunk_size"]
PDF_CHUNK_OVERLAP = PROFILE_CONFIG["chunk_overlap"]
FORCE_REBUILD = PROFILE_CONFIG["force_rebuild"]

print(f"Profile: {PROFILE} | Input: {INPUT_TYPE.upper()} | File: {DATASET_FILE}")

In [ ]:
import nest_asyncio

from src import (
    ExtractedEntity,
    ExtractedRelationship,
    ExtractionResult,
    # GraphRAGExtractor,
    GraphRAGManager,
    GraphRAGQueryEngine,
    GraphRAGSchema,
    # GraphRAGService,
    GraphRAGStore,
)

if GraphRAGSchema.ACTIVE_ONTOLOGY != PROFILE:
    raise RuntimeError("Profile mismatch. Restart the kernel and run from the top.")

nest_asyncio.apply()
print(f"✅ Imports ready; ontology and prompts: {GraphRAGSchema.ACTIVE_ONTOLOGY}")

---
## 2. Configure the Half_PSD Pipeline

The `Half_PSD.llm` settings in `profiles.yaml` select the provider, extraction and
query models, worker count, and community size. The next cell reads these settings;
`USE_QWEN` reflects the configured provider rather than acting as a separate toggle.

The checkpoint, JSON, and HTML use the `half_psd` prefix in `output/`.
Rebuilding and exporting replace that profile's existing artifacts.


In [ ]:
LLM_CONFIG = PROFILE_CONFIG["llm"]
LLM_PROVIDER = LLM_CONFIG["provider"]
USE_QWEN = LLM_PROVIDER == "qwen"
EXTRACTION_MODEL = LLM_CONFIG["extraction_model"]
QUERY_MODEL = LLM_CONFIG["query_model"]
QWEN_BASE_URL = LLM_CONFIG["qwen_base_url"]

OUTPUT_PREFIX = PROFILE_CONFIG["output_prefix"]
OUTPUT_DIR = PROJECT_ROOT / "output"
CHECKPOINT_FILE = OUTPUT_DIR / f"{OUTPUT_PREFIX}_graph_store.pkl"
GRAPH_DATA_FILE = OUTPUT_DIR / f"{OUTPUT_PREFIX}_graph_data.json"
GRAPH_TEMPLATE_FILE = PROJECT_ROOT / "graph_template.html"
GRAPH_OUTPUT_FILE = OUTPUT_DIR / f"{OUTPUT_PREFIX}_graph.html"

MAX_PATHS_PER_CHUNK = LLM_CONFIG["max_paths_per_chunk"]
NUM_WORKERS = LLM_CONFIG["num_workers"]
MAX_CLUSTER_SIZE = LLM_CONFIG["max_cluster_size"]
REQUEST_TIMEOUT = LLM_CONFIG["request_timeout"]
REQUEST_MAX_RETRIES = LLM_CONFIG["request_max_retries"]

manager = GraphRAGManager(
    provider=LLM_PROVIDER,
    extraction_model=EXTRACTION_MODEL,
    query_model=QUERY_MODEL,
    qwen_base_url=QWEN_BASE_URL,
    max_paths_per_chunk=MAX_PATHS_PER_CHUNK,
    num_workers=NUM_WORKERS,
    max_cluster_size=MAX_CLUSTER_SIZE,
    request_timeout=REQUEST_TIMEOUT,
    request_max_retries=REQUEST_MAX_RETRIES,
)

EXTRACTION_LLM = manager.extraction_llm
QUERY_LLM = manager.query_llm

print(
    f"✅ Using {LLM_PROVIDER}: {EXTRACTION_LLM.model} for extraction, "
    f"{QUERY_LLM.model} for querying"
)


---
## 3. Half_PSD Ontology

`ontology.yaml → ontologies → Half_PSD` defines the allowed entity and relationship
types for the DCU use case. These definitions drive the extraction prompt and
Pydantic validation. The next cell displays the active vocabulary.


In [ ]:
ENTITY_TYPES = GraphRAGSchema.ENTITY_TYPES
RELATION_TYPES = GraphRAGSchema.RELATION_TYPES

print("✅ Ontology:")
print(f"   Entity types:       {ENTITY_TYPES}")
print(f"   Relationship types: {RELATION_TYPES}")

---
## 4. Extraction Prompt

The selected profile in prompt.yaml supplies the extraction instructions.
Allowed types and their descriptions are inserted from ontology.yaml.
The LLM returns descriptions alongside entities and relationships so community
summaries can retain context, conditions, and qualifications.

In [ ]:
KG_TRIPLET_EXTRACT_TMPL = GraphRAGSchema.extraction_prompt()

print("✅ Extraction prompt ready")
print(f"\nPreview (first 300 chars):\n{KG_TRIPLET_EXTRACT_TMPL[:300]}...")

---
## 5. Pydantic Extraction Models

Structured output is validated using the selected ontology:

- ExtractedEntity: name, type, description
- ExtractedRelationship: source, target, relation, description
- ExtractionResult: the entity and relationship lists

EntityType and RelationType are runtime Literal types loaded from YAML.
Pydantic rejects labels outside the selected profile.

In [ ]:
print("✅ Pydantic extraction models:")
print("  ", ExtractedEntity.__name__)
print("  ", ExtractedRelationship.__name__)
print("  ", ExtractionResult.__name__)

---
## 6. GraphRAGExtractor

The extractor sends each section chunk to the configured LLM and parses entities
and relationships using the Half_PSD ontology. Entity and relationship descriptions
retain source context for community summaries.

Relationships with endpoints missing from the returned entity list are rejected.
Their details are retained in the exported JSON's `rejected_relationships` list for
review. Worker count and maximum relationships per chunk come from `profiles.yaml`.


In [ ]:
kg_extractor = manager.extractor

print(f"✅ {type(kg_extractor).__name__} ready")
print(f"   Parallel workers: {kg_extractor.num_workers}")
print(f"   Maximum paths per chunk: {kg_extractor.max_paths_per_chunk}")

---
## 7. GraphRAGStore

`GraphRAGStore` holds the extracted entity graph and community summaries.
Community detection uses a weighted NetworkX projection and hierarchical Leiden,
then generates summaries for final communities. Relationship weights influence
clustering; they are not causal confidence scores.

For this demo, inspect how DCU components, operating modes, symptoms, and maintenance
steps group together. Source descriptions and conditions remain necessary when
interpreting a candidate causal relationship.


In [ ]:
# The manager creates this store when a graph is built or loaded.
print(f"✅ Graph store class ready: {GraphRAGStore.__name__}")

---
## 8. GraphRAGQueryEngine

The query engine asks each saved community summary for relevant information, then
aggregates the partial answers with the configured query model. It does not perform
explicit directed-path verification or retrieve PDF excerpts at query time.

Use graph provenance to check the source behind an answer, especially relationship
direction and mode-dependent behavior. A causal label alone does not establish a
confirmed diagnosis.


In [ ]:
# The query engine is instantiated after a graph has been built or loaded.
print(f"✅ Query engine class ready: {GraphRAGQueryEngine.__name__}")

## 9. Inspect the Half_PSD Manual and Section Ranges

Source: `data/152E-GEN-8002(doc)-C.pdf`.

The heading pairs below come from `Half_PSD.section_ranges` in `profiles.yaml`.
Each pair includes its first heading and stops before its second heading.


In [ ]:
if not DATASET_FILE.is_file():
    raise FileNotFoundError(f"PDF not found: {DATASET_FILE}. Supply a text-based PDF for {PROFILE}.")
print(f"PDF: {DATASET_FILE.name}")
if PDF_SECTION_RANGES is not None:
    print(f"Section ranges: {PDF_SECTION_RANGES}")
elif PDF_SECTIONS is not None:
    print(f"Sections: {PDF_SECTIONS}")
else:
    print(f"Pages: {PDF_PAGES if PDF_PAGES is not None else 'all'}")
print(f"Chunk size: {PDF_CHUNK_SIZE} tokens | Overlap: {PDF_CHUNK_OVERLAP}")

## 10. Load Half_PSD Section Chunks

The loader joins each selected section across PDF pages before splitting by token
budget. A section that fits the budget remains one chunk; larger sections use the
configured chunk size and overlap. Separate heading ranges remain separate inputs.

Each chunk retains its section title and source-page provenance. A chunk can have
several page/line entries. Extracted relationships inherit chunk-level evidence;
the line range is not an exact identification of the sentence supporting a triple.


In [ ]:
if GraphRAGSchema.ACTIVE_ONTOLOGY != PROFILE:
    raise RuntimeError("Profile mismatch. Restart the kernel and run from the top.")

nodes = manager.load_pdf_documents(
    DATASET_FILE,
    pages=PDF_PAGES,
    sections=PDF_SECTIONS,
    section_ranges=PDF_SECTION_RANGES,
    chunk_size=PDF_CHUNK_SIZE,
    chunk_overlap=PDF_CHUNK_OVERLAP,
)

if not nodes:
    raise ValueError("The selected input produced no documents.")
print(f"✅ Loaded {len(nodes)} input nodes for {PROFILE}")

In [ ]:
for title, next_title in PDF_SECTION_RANGES:
    section_nodes = [
        node for node in nodes
        if node.metadata.get("section_title") == title
        and node.metadata.get("next_section_title") == next_title
    ]
    source_pages = sorted({
        item["page_number"]
        for node in section_nodes
        for item in node.metadata.get("provenance", [])
    })
    print(f"{title} → {next_title}")
    print(f"  Chunks: {len(section_nodes)} | PDF pages: {source_pages}")


In [ ]:
SAMPLE_INDEX = 0  # Choose a loaded Half_PSD chunk
if not 0 <= SAMPLE_INDEX < len(nodes):
    raise IndexError(f"SAMPLE_INDEX must be between 0 and {len(nodes) - 1}")
sample = nodes[SAMPLE_INDEX]
print(f"Section: {sample.metadata.get('section_title')}")
for location in sample.metadata.get("provenance", []):
    print(f"Page {location['page_number']}, "
          f"lines {location.get('line_start', '?')}–{location.get('line_end', '?')}")
print(sample.text)


---
## 11. Build the Knowledge Graph 

Now we wire everything together and run the extraction pipeline.

`PropertyGraphIndex` handles the full workflow:
1. Passes each chunk to `GraphRAGExtractor`
2. The extractor calls the LLM with our ontology-constrained prompt
3. Parsed entities and relationships are stored in `GraphRAGStore`

-> This is the most time-consuming step!


Half_PSD artifacts are `output/half_psd_graph_store.pkl`,
`output/half_psd_graph_data.json`, and `output/half_psd_graph.html`.

Set `Half_PSD.force_rebuild: true` in `profiles.yaml` when inputs, heading ranges,
chunk settings, models, ontology, or prompts change. Otherwise the saved checkpoint
is reused if present. Restart the kernel after configuration or code changes.
Old checkpoints cannot recover rejections that were never recorded.


In [ ]:
CHECKPOINT_FILE

In [ ]:
REBUILD_GRAPH = FORCE_REBUILD or not CHECKPOINT_FILE.exists()

if REBUILD_GRAPH:
    # Community detection is kept for Section 12 so the original flow remains clear.
    graph_store = manager.build_knowledge_graph(
        documents=nodes,
        build_communities=False,
    )
else:
    graph_store = manager.load_knowledge_graph(CHECKPOINT_FILE)

In [ ]:
# Optional extra LLM call on one loaded document/chunk.
RUN_EXTRACTION_TEST = False
if RUN_EXTRACTION_TEST:
    extraction_test = await manager.atest_extraction(document_index=SAMPLE_INDEX)

In [ ]:
entities_by_type = manager.print_unique_entities()

In [ ]:
# Inspect the DCU when present, otherwise use an available entity.
entity_names = [name for names in entities_by_type.values() for name in names]
ENTITY_TO_INSPECT = "Door Control Unit" if "Door Control Unit" in entity_names else next(
    (name for names in entities_by_type.values() for name in names),
    None,
)
entity_details = (
    manager.inspect_entity(ENTITY_TO_INSPECT)
    if ENTITY_TO_INSPECT is not None else None
)

---
## 12. Build Half_PSD Communities and Summaries

Summaries use `prompt.yaml → prompts → Half_PSD → community_summary`.
They organize the extracted DCU relationships into groups for query answering.
The next cell builds and saves summaries for a rebuilt graph, or reuses those in
the existing Half_PSD checkpoint.


In [ ]:
if REBUILD_GRAPH:
    summaries = graph_store.build_communities(
        summary_llm=EXTRACTION_LLM,
        max_cluster_size=MAX_CLUSTER_SIZE,
    )
    manager.save_knowledge_graph(CHECKPOINT_FILE)
else:
    summaries = graph_store.get_community_summaries()

print(f"\n✅ {len(summaries)} community summaries ready for querying")

## 13. Explore the Half_PSD Graph

From `causalRAG`, run:

```bash
.venv/bin/streamlit run app.py -- half_psd
```

The app loads the Half_PSD checkpoint and supports graph inspection and queries.
The next cell exports `output/half_psd_graph_data.json` and
`output/half_psd_graph.html` using the current template. Edge labels are always
visible in light text. Hover an edge to inspect its source evidence.

The JSON also contains `rejected_relationships` for later review.


In [ ]:
# A running kernel may still hold classes imported before code changes.
if not hasattr(graph_store, "community_members"):
    raise RuntimeError("Restart the kernel and run from the top to export community assignments.")

manager.visualize(
    graph_data_file=GRAPH_DATA_FILE,
    template_file=GRAPH_TEMPLATE_FILE,
    output_file=GRAPH_OUTPUT_FILE,
)

---
## 14. Ask Questions About the Half_PSD System

The following cells use the Half_PSD questions from `profiles.yaml` to explore:

- Components, signals, and parameters connected to the DCU.
- Conditions associated with abnormal doorway behavior.
- Initialization, reset, and replacement preconditions.
- Causal versus diagnostic, control, monitoring, and procedural relationships.

For a cross-section demo, also try: **“After DCU reset or replacement, why is a
successful manual open/close test insufficient to establish readiness for automatic
operation? Connect the checks and mode settings with the normal command path.”**

Compare the answer with the graph and manual evidence. Answers are synthesized
from saved summaries and can reflect errors in extraction or summarization.


In [ ]:
query_engine = GraphRAGQueryEngine(
    graph_store=graph_store,
    community_llm=EXTRACTION_LLM,
    llm=QUERY_LLM,
)

print("✅ Query engine ready")

In [ ]:
q1 = PROFILE_CONFIG["questions"][0]
print(f"Query: {q1}")
print("=" * 70)
print(query_engine.custom_query(q1))

In [ ]:
q2 = PROFILE_CONFIG["questions"][1]
print(f"Query: {q2}")
print("=" * 70)
print(query_engine.custom_query(q2))

In [ ]:
q3 = PROFILE_CONFIG["questions"][2]
print(f"Query: {q3}")
print("=" * 70)
print(query_engine.custom_query(q3))

In [ ]:
# Replace this with your own question about the Half_PSD manual.
your_question = PROFILE_CONFIG["questions"][3]
print(f"Query: {your_question}")
print("=" * 70)
print(query_engine.custom_query(your_question))

In [ ]:
CHECKPOINT_FILE.exists()

---
## 15. Half_PSD Pipeline Reference

| Component | Purpose |
|---|---|
| `data/152E-GEN-8002(doc)-C.pdf` | Source operation and maintenance manual |
| `profiles.yaml → Half_PSD` | Heading ranges, chunking, models, output prefix, questions |
| `ontology.yaml → Half_PSD` | Allowed entity and relationship types |
| `prompt.yaml → Half_PSD` | Extraction, summary, answering, and synthesis instructions |
| `GraphRAGManager` | Load sections, build, persist, inspect, and query |
| `GraphRAGStore` | Entity graph, rejected relationships, and community summaries |
| `output/half_psd_graph_data.json` | Exported graph, provenance, and rejection audit |
| `output/half_psd_graph.html` | Standalone interactive graph |
